# Enterprise Orchestration Lab — GRPO RL Training

GRPO training with environment-grounded rewards. Requires Colab T4 GPU.

> If numpy errors occur, do **Runtime > Factory reset runtime** first.

---
## 1. Install and Validate

In [ ]:
numpy_ok = True
try:
    import numpy as np
    from numpy._core.umath import isalpha
    print(f'numpy {np.__version__} OK')
except Exception as e:
    numpy_ok = False
    import subprocess
    subprocess.check_call(['pip', 'install', '-q', 'numpy==1.26.4'])
    print('numpy fixed. Click Runtime > Restart session, then re-run.')

if numpy_ok:
    import subprocess
    subprocess.check_call(['pip', 'install', '-q', 'trl', 'peft', 'accelerate', 'bitsandbytes', 'datasets', 'matplotlib'])
    subprocess.check_call(['pip', 'install', '-q', 'openenv-core'])
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import get_peft_model, LoraConfig
    from trl import GRPOTrainer, GRPOConfig
    from datasets import Dataset
    print(f'torch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
    if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')
    print('All imports OK.')

---
## 2. Clone and Setup Environment

In [ ]:
import os, sys, json, re, random
if not os.path.exists('scaler-final'):
    !git clone https://github.com/redhatsam09/scaler-final.git
if not os.getcwd().endswith('scaler-final'): os.chdir('scaler-final')
!pip install -q -r requirements.txt
if '.' not in sys.path: sys.path.insert(0, '.')
from src.environment import DataCleaningEnv
from src.graders import EnterpriseOrchestrationGrader, MissingValuesGrader
from src.models import Action
env = DataCleaningEnv(seed=42)
obs = env.reset(task_id='task_enterprise_orchestration', seed=42)
print(f'Environment OK: {obs.dataset_shape[0]}x{obs.dataset_shape[1]}, actions={obs.available_actions}')

---
## 3. Define Reward Function (Multi-Level Scoring)

Granular rewards ensure different completions get different scores, which is critical for GRPO to compute non-zero advantages.

In [ ]:
VALID_ACTIONS = {'analyze', 'impute', 'deduplicate', 'validate', 'report_findings',
                 'delegate', 'resolve_alert', 'reconcile_apps', 'oversight_review',
                 'inspect_actor', 'audit_records', 'request_policy_clarification'}
REQUIRED_KEYS = {'action_type', 'target_columns', 'parameters', 'reasoning'}

SYSTEM_PROMPT = """You are an enterprise workflow orchestrator managing CRM, Billing, and Support systems.
Given the current state, output a JSON action with:
- action_type: one of [analyze, impute, deduplicate, validate, report_findings, delegate, resolve_alert, reconcile_apps, oversight_review, inspect_actor, audit_records, request_policy_clarification]
- target_columns: list of column names
- parameters: dict of action parameters
- reasoning: why you chose this action (minimum 15 chars)

Respond ONLY with valid JSON."""

def build_prompt(obs):
    return f"""{SYSTEM_PROMPT}

Current State:
{obs.natural_language_observation}

Dataset: {obs.dataset_shape[0]} rows, {obs.dataset_shape[1]} cols
Missing values: {json.dumps(dict(list(obs.missing_values.items())[:6]))}
Step: {obs.step_count}
Policy version: {obs.policy_version}
KPIs: {json.dumps(obs.kpi_snapshot)}
Actor messages: {obs.actor_messages[-2:] if obs.actor_messages else []}
Urgency: {obs.urgency_signals if obs.urgency_signals else 'None'}

What action should you take?"""


# Global reward log for plotting training curves
reward_log = []

def environment_reward_function(completions, prompts, **kwargs):
    """
    Multi-level reward function for GRPO. Assigns granular scores so
    different completions get different rewards (critical for non-zero loss).
    
    Scoring levels:
      -1.0  no JSON found at all
      -0.7  JSON found but unparseable
      -0.3  parsed but missing required keys
      -0.1  valid keys but invalid action_type
       0.0  valid action_type but short/missing reasoning
      +env  environment step reward + grader score (0.1 to 1.0)
    
    Small noise (+-0.02) is added to break ties between similar outputs.
    """
    rewards = []
    reward_env = DataCleaningEnv(seed=42)
    task_ids = ['task_enterprise_orchestration', 'task_missing_values',
                'task_duplicate_handling', 'task_complex_validation']

    for i, completion in enumerate(completions):
        noise = random.uniform(-0.02, 0.02)  # Break ties
        
        # Level 0: No JSON at all
        match = re.search(r'\{.*\}', completion, re.DOTALL)
        if not match:
            rewards.append(-1.0 + noise)
            continue
        
        # Level 1: JSON found but unparseable
        try:
            data = json.loads(match.group())
        except json.JSONDecodeError:
            rewards.append(-0.7 + noise)
            continue
        
        # Level 2: Parsed but missing keys
        present_keys = set(data.keys()) & REQUIRED_KEYS
        if len(present_keys) < len(REQUIRED_KEYS):
            key_score = len(present_keys) / len(REQUIRED_KEYS)  # 0.0 to 0.75
            rewards.append(-0.3 + 0.2 * key_score + noise)
            continue
        
        # Level 3: Invalid action_type
        action_type = data.get('action_type', '')
        if action_type not in VALID_ACTIONS:
            rewards.append(-0.1 + noise)
            continue
        
        # Level 4: Valid structure — bonus for reasoning length
        reasoning = data.get('reasoning', '')
        reasoning_bonus = min(0.1, len(reasoning) / 150.0 * 0.1)
        
        # Level 5: Run through actual environment
        try:
            task_id = task_ids[i % len(task_ids)]
            obs = reward_env.reset(task_id=task_id, seed=1000 + i)
            action = Action(
                action_type=action_type,
                target_columns=data.get('target_columns', obs.column_names[:3]),
                parameters=data.get('parameters', {}),
                reasoning=reasoning if len(reasoning) >= 15 else 'Auto-generated reasoning text',
            )
            _, reward, _, _ = reward_env.step(action)
            grader = EnterpriseOrchestrationGrader if 'enterprise' in task_id else MissingValuesGrader
            grade = grader.grade(reward_env.current_episode)
            env_score = 0.5 * reward.value + 0.3 * float(grade) + reasoning_bonus
            rewards.append(min(1.0, max(-1.0, env_score + noise)))
        except Exception:
            rewards.append(0.0 + reasoning_bonus + noise)
    
    # Log batch mean for plotting
    if rewards:
        reward_log.append(sum(rewards) / len(rewards))
    
    return rewards

# Validate
tests = [
    ('no json', 'hello world'),
    ('bad json', '{not valid json}'),
    ('missing keys', json.dumps({'action_type': 'analyze'})),
    ('bad action', json.dumps({'action_type': 'fly', 'target_columns': [], 'parameters': {}, 'reasoning': 'testing invalid'})),
    ('valid', json.dumps({'action_type': 'analyze', 'target_columns': ['account_id'], 'parameters': {}, 'reasoning': 'Profile quality before changes'})),
]
for label, comp in tests:
    r = environment_reward_function([comp], ['test'])[0]
    print(f'  {label:15s} -> reward = {r:+.4f}')
reward_log.clear()  # Clear test entries
print('Reward function validated: different inputs produce different scores.')

---
## 4. Load Model (4-bit LoRA)

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map='auto')
lora_config = LoraConfig(r=16, lora_alpha=16, lora_dropout=0, task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
model = get_peft_model(model, lora_config)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print(f'Model loaded: {MODEL_NAME}')
model.print_trainable_parameters()

---
## 5. Generate Training Prompts

In [ ]:
prompt_env = DataCleaningEnv(seed=42)
prompts = []
task_ids = ['task_enterprise_orchestration', 'task_missing_values',
            'task_duplicate_handling', 'task_complex_validation']
for i in range(50):
    obs = prompt_env.reset(task_id=task_ids[i % len(task_ids)], seed=2000 + i)
    prompts.append(build_prompt(obs))
dataset = Dataset.from_dict({'prompt': prompts})
print(f'Created {len(dataset)} prompts (avg {sum(len(p) for p in prompts)//len(prompts)} chars)')

---
## 6. Run GRPO Training

Temperature 0.9 ensures diverse generations. 30 steps, ~20 min on T4.

In [ ]:
reward_log.clear()

config = GRPOConfig(
    output_dir='/content/grpo_checkpoint',
    num_generations=4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    max_steps=30,
    logging_steps=1,
    save_strategy='no',
    max_completion_length=512,
    temperature=0.9,
    report_to=[],
)

trainer = GRPOTrainer(
    model=model, tokenizer=tokenizer, config=config,
    train_dataset=dataset, reward_funcs=[environment_reward_function],
)

print('Starting GRPO training...')
result = trainer.train()
print(f'\nTraining complete. Loss: {result.training_loss:.6f}, Steps: {result.global_step}')

---
## 7. Training Curves (Loss + Reward)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.facecolor': '#0f172a',
    'axes.facecolor': '#1e293b', 'axes.edgecolor': '#334155',
    'text.color': '#f8fafc', 'axes.labelcolor': '#94a3b8',
    'xtick.color': '#64748b', 'ytick.color': '#64748b'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Training loss from trainer logs
log_history = trainer.state.log_history
loss_steps = [h['step'] for h in log_history if 'loss' in h]
loss_vals = [h['loss'] for h in log_history if 'loss' in h]
if loss_steps:
    axes[0].plot(loss_steps, loss_vals, color='#38bdf8', linewidth=2, marker='o', markersize=4)
    axes[0].fill_between(loss_steps, loss_vals, alpha=0.15, color='#38bdf8')
axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('GRPO Training Loss', color='#38bdf8', fontweight='bold')
axes[0].grid(True, alpha=0.2, color='#475569')

# Plot 2: Mean reward per batch from our reward_log
if reward_log:
    axes[1].plot(range(1, len(reward_log)+1), reward_log, color='#22c55e', linewidth=2, marker='o', markersize=3)
    axes[1].fill_between(range(1, len(reward_log)+1), reward_log, alpha=0.15, color='#22c55e')
    # Add trend line
    if len(reward_log) > 3:
        import numpy as np
        z = np.polyfit(range(len(reward_log)), reward_log, 1)
        p = np.poly1d(z)
        axes[1].plot(range(1, len(reward_log)+1), p(range(len(reward_log))),
                     '--', color='#f59e0b', linewidth=1.5, alpha=0.8, label=f'Trend (slope={z[0]:.4f})')
        axes[1].legend(facecolor='#1e293b', edgecolor='#334155')
axes[1].set_xlabel('Reward Batch')
axes[1].set_ylabel('Mean Reward')
axes[1].set_title('Mean Reward per Batch', color='#22c55e', fontweight='bold')
axes[1].axhline(y=0, color='#64748b', linestyle='--', alpha=0.4)
axes[1].grid(True, alpha=0.2, color='#475569')

plt.tight_layout()
plt.savefig('artifacts/training_curves.svg', facecolor='#0f172a')
plt.savefig('artifacts/training_curves.png', facecolor='#0f172a', dpi=150)
plt.show()
print(f'Curves saved. Loss points: {len(loss_steps)}, Reward batches: {len(reward_log)}')

---
## 8. Evaluate Trained Model

In [ ]:
eval_env = DataCleaningEnv(seed=99)
eval_rewards = []
model.eval()
for i in range(12):
    task_id = task_ids[i % len(task_ids)]
    obs = eval_env.reset(task_id=task_id, seed=5000 + i)
    prompt = build_prompt(obs)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1536).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True)
    completion = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    reward = environment_reward_function([completion], [prompt])[0]
    eval_rewards.append(reward)
    print(f'Episode {i+1:2d} | {task_id:36s} | reward = {reward:+.4f}')

mean_reward = sum(eval_rewards) / len(eval_rewards)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#22c55e' if r > 0 else '#ef4444' for r in eval_rewards]
axes[0].bar(range(1, 13), eval_rewards, color=colors)
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward')
axes[0].set_title('Per-Episode Eval Rewards', color='#38bdf8', fontweight='bold')
axes[0].axhline(y=0, color='#64748b', linestyle='--', alpha=0.4)
axes[0].grid(True, alpha=0.2, color='#475569')
axes[1].bar(['Trained'], [mean_reward], color='#3b82f6', width=0.4)
axes[1].set_ylabel('Mean Reward')
axes[1].set_title(f'Mean Reward: {mean_reward:+.4f}', color='#3b82f6', fontweight='bold')
axes[1].set_ylim(-0.5, 1.0)
axes[1].grid(True, alpha=0.2, color='#475569')
plt.tight_layout()
plt.savefig('artifacts/eval_results.svg', facecolor='#0f172a')
plt.show()
print(f'Mean evaluation reward: {mean_reward:+.4f}')

---
## 9. Save Metrics

In [ ]:
from pathlib import Path
metrics = {
    'model': MODEL_NAME, 'quantization': '4bit_nf4', 'lora_r': 16,
    'training': {'mode': 'grpo', 'steps': result.global_step, 'loss': float(result.training_loss),
                 'reward_log': [round(r, 4) for r in reward_log]},
    'evaluation': {'mean_reward': round(mean_reward, 4),
                   'per_episode': [round(r, 4) for r in eval_rewards]},
}
Path('artifacts').mkdir(exist_ok=True)
Path('artifacts/grpo_training_metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))